In [142]:
import pandas as pd
import subprocess
from functools import reduce
import numpy as np

In [ ]:
admixture_dir = '/gpfs/commons/datasets/1000genomes/release-20130502-supporting/admixture_files'

## Explore 1000 genomes data

In [8]:
populations = pd.read_csv('/gpfs/commons/datasets/1000genomes/phase3/20131219.populations.tsv',sep='\t',engine='python')
superpopulations = pd.read_csv('/gpfs/commons/datasets/1000genomes/phase3/20131219.superpopulations.tsv', sep='\t',engine='python')

In [10]:
populations.groupby('Super Population')['Final Phase Samples'].sum()

Super Population
AFR    669.0
AMR    352.0
EAS    515.0
EUR    505.0
SAS    494.0
Name: Final Phase Samples, dtype: float64

Choose K=5 for admixture files; read in .ped and .map files

In [19]:
K=5 # admixture model assumes 5 ancestral populations (more coarse resolution)

In [ ]:
ped = pd.read_csv(f"{admixture_dir}/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05.ped", sep="\s+", header=None,usecols=range(6),names=['FID','IID','PID','MID','Sex','Phenotype'])
ped.shape

(2504, 6)

In [38]:
admixture_ancestry_fractions = pd.read_csv(f"{admixture_dir}/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05.{K}.Q", sep='\s+',header=None,names=range(K))
admixture_ancestry_fractions

,0,1,2,3,4
0,0.372130,0.627840,0.000010,0.000010,0.000010
1,0.382685,0.617285,0.000010,0.000010,0.000010
2,0.415996,0.583974,0.000010,0.000010,0.000010
3,0.389080,0.610890,0.000010,0.000010,0.000010
4,0.407496,0.592474,0.000010,0.000010,0.000010
...,...,...,...,...,...
2499,0.000010,0.000010,0.069528,0.913966,0.016486
2500,0.000010,0.000010,0.058804,0.921309,0.019867
2501,0.000010,0.000010,0.053577,0.913508,0.032896
2502,0.000010,0.000010,0.027355,0.894077,0.078549


In [36]:
map = pd.read_csv(f"{admixture_dir}/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05.map", sep='\s+', header=None,
                     names=["CHR","ID","GEN_DIST","BP"])
map.index

RangeIndex(start=0, stop=193634, step=1)

In [51]:
admixture_af = pd.read_csv(f"{admixture_dir}/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05.{K}.P", sep='\s+',header=None,names=range(K))
admixture_af.index

RangeIndex(start=0, stop=193634, step=1)

In [52]:
admixture_af = pd.concat([map,admixture_af],axis=1)

In [43]:
# variance for binomial distribution symmetric around 0.5 so doesn't matter much whether major or minor allele
for i in range(5):
    print(admixture_af[i].min())
    print(admixture_af[admixture_af[i]<0.5].shape)

1e-05
(23914, 5)
1e-05
(23879, 5)
0.061877
(14890, 5)
0.094704
(12926, 5)
0.024658
(18666, 5)


In [53]:
admixture_af['af_variance'] = admixture_af[range(K)].var(axis=1)
admixture_af.sort_values('af_variance',ascending=False)

,CHR,ID,GEN_DIST,BP,0,1,2,3,4,af_variance
141117,12,12:85808667,0,85808667,0.053860,0.016563,0.933379,0.918828,0.882201,0.230870
20952,2,2:72457091,0,72457091,0.007649,0.000010,0.645624,0.921774,0.974494,0.229038
98857,8,8:11922512,0,11922512,0.025122,0.033879,0.815348,0.891302,0.981013,0.228636
188360,20,20:62178105,0,62178105,0.000010,0.002521,0.940283,0.495336,0.966393,0.226762
154585,14,14:57835368,0,57835368,0.044583,0.043045,0.966963,0.899899,0.849785,0.224505
...,...,...,...,...,...,...,...,...,...,...
145716,13,13:31874223,0,31874223,0.788444,0.792776,0.785299,0.796312,0.790683,0.000018
51989,4,4:72542419,0,72542419,0.900885,0.902139,0.904835,0.905565,0.911559,0.000017
123836,10,10:104757709,0,104757709,0.804774,0.811418,0.812382,0.808412,0.813141,0.000012
93358,7,7:93207873,0,93207873,0.927293,0.923569,0.919931,0.924169,0.928686,0.000012


In [59]:
admixture_af['af_var_decile'] = (pd.qcut(admixture_af['af_variance'], 10, labels=False))/10 # discretize into equal size buckets based on deciles
admixture_af.sort_values('af_variance',ascending=False)
# af_var_decile = 0 --> point in dataset below which 10% of the data falls
# af_var_decile = 9 --> point in dataset above which 10% of the data falls



,CHR,ID,GEN_DIST,BP,0,1,2,3,4,af_variance,af_var_decile
141117,12,12:85808667,0,85808667,0.053860,0.016563,0.933379,0.918828,0.882201,0.230870,0.9
20952,2,2:72457091,0,72457091,0.007649,0.000010,0.645624,0.921774,0.974494,0.229038,0.9
98857,8,8:11922512,0,11922512,0.025122,0.033879,0.815348,0.891302,0.981013,0.228636,0.9
188360,20,20:62178105,0,62178105,0.000010,0.002521,0.940283,0.495336,0.966393,0.226762,0.9
154585,14,14:57835368,0,57835368,0.044583,0.043045,0.966963,0.899899,0.849785,0.224505,0.9
...,...,...,...,...,...,...,...,...,...,...,...
145716,13,13:31874223,0,31874223,0.788444,0.792776,0.785299,0.796312,0.790683,0.000018,0.0
51989,4,4:72542419,0,72542419,0.900885,0.902139,0.904835,0.905565,0.911559,0.000017,0.0
123836,10,10:104757709,0,104757709,0.804774,0.811418,0.812382,0.808412,0.813141,0.000012,0.0
93358,7,7:93207873,0,93207873,0.927293,0.923569,0.919931,0.924169,0.928686,0.000012,0.0


In [161]:
def sun_generate_sim_data(root_dir,intermediate_file_dir,intermediate_file_suffix,ps,num_markers_assoc,e,extra_subgroups_size, K=5):
    '''
    Generate synthetic data similar to Sun et al. (Multi-view biclustering for genotype-phenotype association studies of complex diseases)
    using 1000 Genomes Phase 3 data. Use admixture files which contain 193634 markers with MAF>5% and 2504 individuals. 

    PARAMS:
    root_dir: root directory for 1000 Genomes data
    intermediate_file_dir: dir to write intermediate files to (when using plink for example)
    intermediate_file_suffix: such that if multiple simulations are created, each is distinctly defined
    M: number of clinical features (right now assuming all from one domain & all binary)
    ps: variable controlling how much population stratification is affecting geno-pheno relationship (needs to be in range(0,1,size=0.1)) 
    (0-> pick SNPs in bottom 10% by allele frequency variance i.e. little pop. strat., 0.9-> pick SNPS in top 10% by allele frequency variance i.e. large pop. strat.)
    num_markers_assoc: number of markers with an associated with subtype classification (if rij>int(0.4*markers_assoc) then subject i in subgroup j)
    e: relative effect that genetic variation contributed to the effect of the phenotype. e in [0,1]. (decreased e means higher level of disagreement between genotypic and phenotypic subgroups)
    num_clinical_assoc: number of clinical features associated with subtype classification
    extra_subgroups_size: number of people in s3 and s4 (selected at random)
    K: number of admixture groups to estimate af variance over (K in [5,26] per 1000 genomes phase 3 paper)
    '''

    # 1. Read in allele frequencies per 5 admixture groups to estimate af variance across groups
    admixture_af = pd.read_csv(f"{root_dir}/release-20130502-supporting/admixture_files/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05.{K}.P", sep='\s+',header=None,names=range(K))
    map = pd.read_csv(f"{root_dir}/release-20130502-supporting/admixture_files/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05.map", sep='\s+', header=None,
                        names=["CHR","ID","GEN_DIST","BP"])
    # add SNP information to admixture af df
    admixture_af = pd.concat([map,admixture_af],axis=1)
    admixture_af['af_variance'] = admixture_af[range(K)].var(axis=1)
    admixture_af['af_var_decile'] = (pd.qcut(admixture_af['af_variance'], 10, labels=False))/10 # discretize into equal size buckets based on deciles

    # 2. Generate genetic subgroups
    genetic_subgroups = []
    for genetic_subgroup in range(2):
        # Select SNP group based on num_markers_assoc and ps
        assert admixture_af[admixture_af['af_var_decile']==ps].shape[0]>num_markers_assoc, f"number of genetic features assoc. ({num_markers_assoc}) is too large, only {admixture_af[admixture_af['af_var_decile']==ps].shape[0]} markers in decile {ps} group"
        markers_assoc = admixture_af[admixture_af['af_var_decile']==ps].sample(n=num_markers_assoc, replace=False)['ID'].values.tolist()
        assert len(set(markers_assoc))==len(markers_assoc) # make sure ped file has unique rows

        
        # extract selected markers
        with open(f'{intermediate_file_dir}/markers_assoc_g{genetic_subgroup}_{intermediate_file_suffix}.txt','w') as f:
            for snp in markers_assoc:
                f.write(snp + "\n")
        # major allele set to A2 (If a binary fileset was originally loaded, --keep-allele-order forces the original A1/A2 allele encoding to be preserved; otherwise, the major allele is set to A2)
        plink_extract = f'''
        module load plink/1.9 && plink --file {root_dir}/release-20130502-supporting/admixture_files/ALL.wgs.phase3_shapeit2_filtered.20141217.maf0.05 \
            --extract {intermediate_file_dir}/markers_assoc_g{genetic_subgroup}_{intermediate_file_suffix}.txt \
            --make-bed \
            --out {intermediate_file_dir}/subset_markers_g{genetic_subgroup}_{intermediate_file_suffix}
        '''
        result = subprocess.run(plink_extract, shell=True, check=True, executable="/bin/bash")
        # get marker values for each individual (0 - no copies of minor allele, 1 - 1 copy of minor allele, 2 - 2 copies of minor allele)
        plink_count = f'''
        module load plink/1.9 && \
            plink --bfile {intermediate_file_dir}/subset_markers_g{genetic_subgroup}_{intermediate_file_suffix} \
                --recode A \
                --out {intermediate_file_dir}/subset_markers_g{genetic_subgroup}_{intermediate_file_suffix}'''
        result = subprocess.run(plink_count, shell=True, check=True, executable="/bin/bash")

        raw = pd.read_csv(f'{intermediate_file_dir}/subset_markers_g{genetic_subgroup}_{intermediate_file_suffix}.raw',sep="\s+")
        geno = raw.drop(columns=['FID','IID','PAT','MAT','SEX','PHENOTYPE'])
        # assert they are all ≤ 0.5 (i.e., A1 is the minor allele)
        assert (geno.sum(axis=0) / (2 * geno.shape[0]) <= 0.5).all(), "Some SNPs have A1 frequency > 0.5"
        geno_cols = [col for col in raw.columns if not col in ['FID','IID','PAT','MAT','SEX','PHENOTYPE']]
        assert len(geno_cols) == num_markers_assoc
        raw[geno_cols] = (raw[geno_cols] > 0).astype(int) # recode s.t. values 1 and 2 map to 1
        genetic_subgroup_df = raw.set_index('IID')[geno_cols].sum(axis=1).reset_index(name=f'r')
        genetic_subgroup_df['genetic_subgroup'] = genetic_subgroup
        
        deciles, bins = pd.qcut(genetic_subgroup_df["r"], 10, labels=False, retbins=True)
        genetic_subgroup_df[f'subgroup'] =genetic_subgroup_df['r']>bins[-3] # Top 20% of people per r
        genetic_subgroups.append(genetic_subgroup_df)
    genetic_subgroups = pd.concat(genetic_subgroups)

    # 3. Generate phenotypic subgroups
    # can just use bins[-4] - somewhat equivalent to 7.5
    phenotypic_subgroups = []
    for phenotypic_subgroup in range(2):
        phenotypic_subgroup_df = genetic_subgroups[genetic_subgroups['genetic_subgroup']==phenotypic_subgroup][['IID','r']].copy()
        phenotypic_subgroup_df['phenotypic_subgroup'] = phenotypic_subgroup
        phenotypic_subgroup_df['subgroup'] = phenotypic_subgroup_df['r']*e + np.random.randn(len(phenotypic_subgroup_df)) > bins[-4]*e
        phenotypic_subgroups.append(phenotypic_subgroup_df)
    for phenotypic_subgroup in range(2,4):
        # randomly select extra_subgroups_size people
        randomly_selected = pd.Series(genetic_subgroups.IID.unique()).sample(extra_subgroups_size).values.tolist()
        phenotypic_subgroup_df = genetic_subgroups[genetic_subgroups['genetic_subgroup']==phenotypic_subgroup][['IID','r']].copy()
        phenotypic_subgroup_df['phenotypic_subgroup'] = phenotypic_subgroup
        phenotypic_subgroup_df['subgroup'] = phenotypic_subgroup_df['IID'] in randomly_selected
        phenotypic_subgroups.append(phenotypic_subgroup_df)
    phenotypic_subgroups = pd.concat(phenotypic_subgroups)


    return genetic_subgroups, phenotypic_subgroups
    


In [9]:
root_dir='/gpfs/commons/datasets/1000genomes'

In [ ]:
genetic_subgroups,phenotypic_subgroups = sun_generate_sim_data(root_dir='/gpfs/commons/datasets/1000genomes', intermediate_file_dir='/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno/data/simulations', intermediate_file_suffix='', 
                                      ps=0, num_markers_assoc=2000, e=0.5,extra_subgroups_size=200, K=5)


In [140]:
e = 0.5

In [ ]:
for phenotypic_subgroup in range(2):
    genetic_subset = genetic_subgroups[genetic_subgroups['genetic_subgroup']==phenotypic_subgroup].copy()
    genetic_subset['r']*e

In [141]:
phenotypic_subgroup = 0

In [145]:
bins[-4]*e

379.0

In [146]:
genetic_subset

,IID,r,genetic_subgroup,subgroup
0,HG02922,484,0,False
1,HG02923,458,0,False
2,HG02938,450,0,False
3,HG02941,443,0,False
4,HG02943,457,0,False
...,...,...,...,...
2499,HG04106,470,0,False
2500,HG04107,499,0,True
2501,HG04210,510,0,True
2502,HG04227,434,0,False


In [148]:
phenotypic_subgroup_df = genetic_subgroups[genetic_subgroups['genetic_subgroup']==phenotypic_subgroup][['IID','r']].copy()
phenotypic_subgroup_df['phenotypic_subgroup'] = phenotypic_subgroup
phenotypic_subgroup_df['subgroup'] = phenotypic_subgroup_df['r']*e + np.random.randn(len(genetic_subset)) > bins[-4]*e
phenotypic_subgroup_df

,IID,r,phenotypic_subgroup,subgroup
0,HG02922,484,0,False
1,HG02923,458,0,False
2,HG02938,450,0,False
3,HG02941,443,0,False
4,HG02943,457,0,False
...,...,...,...,...
2499,HG04106,470,0,False
2500,HG04107,499,0,False
2501,HG04210,510,0,False
2502,HG04227,434,0,False
